# 03 · Agentic compliance pipeline — real data, real model, no fine-tuning

The 4-stage architecture, built on **real datasets end to end**, no hand-written text anywhere:

```
1. Load + classify    -> ~300 REAL, atomic requirements combined from two public HF datasets: EU AI
                         Act obligations (per article/paragraph, with a real legal quote) and GDPR
                         articles (decomposed into their own key obligations). Classified in batches
                         as deterministic (rule-checkable) or llm_judgment (needs interpretation),
                         and saved to regulations_dataset.json - nothing is hardcoded in the notebook.
2. Collect evidence    -> a REAL codebase (19k real files, 3k+ repos, 18 languages/IaC types, from
                         the LLM-SAST dataset) searched by TF-IDF retrieval (a small, real RAG) for
                         the ambiguous requirements; a generic pattern check for a few deterministic
                         ones. No synthetic/hand-written snippets anywhere.
3. Verdict + guardrail -> deterministic checks run directly (regex, no LLM); the rest are judged by
                         the LLM in small batches - it gets each requirement + ONLY its own retrieved
                         evidence, then every citation is verified against what was actually shown -
                         an unverifiable citation is downgraded to insufficient_evidence, never trusted.
4. Audit report        -> one row per requirement: verdict + evidence + rationale, checkpointed
                         during the run and saved to disk at the end.
```

**Data sources (all real, all public, loaded live from the Hub):**
- EU AI Act: [`safelegalaidata/eu-ai-act-structured`](https://huggingface.co/datasets/safelegalaidata/eu-ai-act-structured) (subset `obligations`) — 677 atomic obligations, each tied to an exact article/paragraph with a real legal quote, CC-BY-4.0.
- GDPR: [`AYI-NEDJIMI/gdpr-en`](https://huggingface.co/datasets/AYI-NEDJIMI/gdpr-en) — 57 real articles, each with its own `key_obligations` list (152 atomic obligations total), Apache-2.0.
- Evidence: [`aioutfitters/llm-sast-v1`](https://huggingface.co/datasets/aioutfitters/llm-sast-v1) — 19,024 real files from 3,055 repos, Apache-2.0.

By default this samples **200 EU AI Act + 100 GDPR = 300 requirements** (`MAX_AI_ACT_REQUIREMENTS` /
`MAX_GDPR_REQUIREMENTS` in the config cell below — raise them for more, up to 677 + 152 = 829 total).

**`RUNTIME = "kaggle"` by default** — DeepSeek-R1-8B loaded directly via Unsloth on a real T4
GPU, prompting only, no training (same install fixes already proven in the fine-tuning notebooks:
torch pinning, the `huggingface_hub`/`additional_chat_templates` workaround). **Kaggle settings:
Accelerator = GPU T4 x2, Internet = On.** `RUNTIME = "ollama"` runs the same DeepSeek locally via
Ollama instead. `RUNTIME = "mock"` is a rough, generic-keyword stand-in for a quick offline sanity
check of the pipeline logic only - its numbers are not meaningful, only the control flow is.

**At 300 requirements, expect a long run** (likely over an hour on a T4, most of it the LLM-judgment
batches). Verdicts are checkpointed to `agentic_run/verdicts_checkpoint.json` after every batch, so
an interrupted run does not lose completed work. Consider starting with smaller
`MAX_AI_ACT_REQUIREMENTS` / `MAX_GDPR_REQUIREMENTS` (e.g. 20 + 10) for a first end-to-end smoke test
on real Kaggle hardware before committing to a full 300+ run.

This is not legal advice: for a real audit, a qualified compliance/legal reviewer must still check
the borderline verdicts - consistent with this pipeline's own anti-hallucination, human-in-the-loop
design for anything the model cannot ground in actual retrieved evidence.

In [1]:
import os
from pathlib import Path

RUNTIME = "kaggle"       # "kaggle" (real GPU, default) | "ollama" (local laptop) | "mock" (offline smoke test)
OLLAMA_MODEL = "deepseek-r1:8b"   # try "deepseek-r1:1.5b" if 8b is too slow on a 4 GB-VRAM laptop
OLLAMA_URL = "http://localhost:11434/api/generate"

KAGGLE_MODEL = "unsloth/DeepSeek-R1-0528-Qwen3-8B"  # same preset as the fine-tuning notebooks
KAGGLE_MAX_SEQ_LEN = 8192        # headroom for a reasoning model's <think> block + a batched JSON reply
KAGGLE_MAX_NEW_TOKENS = 3072
KAGGLE_STOP = "<\uff5cend\u2581of\u2581sentence\uff5c>"

AI_ACT_DATASET = "safelegalaidata/eu-ai-act-structured"   # 677 atomic obligations, per article/paragraph
GDPR_DATASET = "AYI-NEDJIMI/gdpr-en"                       # 57 articles, 152 atomic key_obligations
MAX_AI_ACT_REQUIREMENTS = 200    # of 677 available - raise for more (up to 677)
MAX_GDPR_REQUIREMENTS = 100      # of 152 available - raise for more (up to 152)
CLASSIFY_BATCH_SIZE = 25         # requirements classified per LLM call
VERDICT_BATCH_SIZE = 4           # requirements judged per LLM call

EVIDENCE_DATASET = "aioutfitters/llm-sast-v1"
EVIDENCE_LIMIT = 500      # how many real LLM-SAST files form the searchable evidence corpus
TOP_K_EVIDENCE = 3        # how many retrieved files are handed to the judge per requirement

RUN_DIR = Path("agentic_run")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUNTIME={RUNTIME!r} | outputs -> {RUN_DIR.resolve()}")
print(f"Target: {MAX_AI_ACT_REQUIREMENTS} EU AI Act + {MAX_GDPR_REQUIREMENTS} GDPR = "
      f"{MAX_AI_ACT_REQUIREMENTS + MAX_GDPR_REQUIREMENTS} requirements")

RUNTIME='kaggle' | outputs -> /kaggle/working/agentic_run
Target: 200 EU AI Act + 100 GDPR = 300 requirements


In [2]:
if RUNTIME == "kaggle":
    print("=== Installing dependencies (unsloth, trl, transformers...) - a few minutes ===")
    import subprocess

    def sh(cmd):
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            print(result.stdout[-3000:], result.stderr[-3000:], sep="\n")
            raise RuntimeError(f"Command failed: {cmd}")

    # Same pinning as the fine-tuning notebooks: installing unsloth unpinned can drag in a CPU-only
    # or mismatched-CUDA torch build from PyPI, replacing Kaggle's own working one.
    try:
        import numpy, PIL, torch, torchvision
        pins = (f"numpy=={numpy.__version__} pillow=={PIL.__version__} torch=={torch.__version__.split('+')[0]} "
                f"torchvision=={torchvision.__version__.split('+')[0]}")
    except Exception:
        pins = "numpy pillow torch torchvision"
    sh("pip install --upgrade -qqq uv")
    sh(f"uv pip install -qqq {pins} bitsandbytes xformers unsloth")
    sh('uv pip install -qqq triton "huggingface_hub>=1.0.0" "datasets==4.3.0"')
    sh('uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"')
    sh("uv pip install -qqq transformers==4.57.6")
    sh("uv pip install -qqq --no-deps trl==0.22.2")
    print("Installation done.")
else:
    print(f"RUNTIME={RUNTIME!r}: skipping the Kaggle install (only needed for RUNTIME='kaggle').")


=== Installing dependencies (unsloth, trl, transformers...) - a few minutes ===
Installation done.


In [3]:
if RUNTIME == "kaggle":
    import torch

    # Same upstream-bug workaround as the fine-tuning notebooks: some transformers/huggingface_hub
    # combinations raise instead of returning [] when a repo has no additional_chat_templates folder
    # (github.com/huggingface/transformers/issues/41813).
    try:
        import transformers.tokenization_utils_base as _tok_utils_base
        import transformers.utils.hub as _hf_hub_utils

        _original_list_repo_templates = _hf_hub_utils.list_repo_templates

        def _safe_list_repo_templates(*args, **kwargs):
            try:
                return _original_list_repo_templates(*args, **kwargs)
            except Exception as exc:
                print(f"list_repo_templates lookup failed ({exc!r}); assuming none found.")
                return []

        _hf_hub_utils.list_repo_templates = _safe_list_repo_templates
        if hasattr(_tok_utils_base, "list_repo_templates"):
            _tok_utils_base.list_repo_templates = _safe_list_repo_templates
    except Exception as exc:
        print(f"Could not patch list_repo_templates ({exc!r}); continuing without the workaround.")

    from unsloth import FastLanguageModel

    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}, {gpu.total_memory / 2**30:.1f} GB")
    if gpu.major < 7:
        raise RuntimeError("Unsloth needs a GPU with CUDA capability 7.0+. On Kaggle choose "
                          "'GPU T4 x2', not P100.")

    kaggle_model, kaggle_tokenizer = FastLanguageModel.from_pretrained(
        model_name=KAGGLE_MODEL, max_seq_length=KAGGLE_MAX_SEQ_LEN, load_in_4bit=True,
        full_finetuning=False,
    )
    FastLanguageModel.for_inference(kaggle_model)  # inference-only - no LoRA adapters, no training
    if kaggle_tokenizer.pad_token is None:
        kaggle_tokenizer.pad_token = kaggle_tokenizer.eos_token
    KAGGLE_STOP_IDS = [i for i in (kaggle_tokenizer.convert_tokens_to_ids(t) for t in [KAGGLE_STOP])
                      if isinstance(i, int) and i != kaggle_tokenizer.unk_token_id]
    print("DeepSeek loaded for direct Kaggle inference (no training).")
else:
    kaggle_model = kaggle_tokenizer = KAGGLE_STOP_IDS = None
    print(f"RUNTIME={RUNTIME!r}: skipping Kaggle model loading.")


/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:2161: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


GPU: Tesla T4, 14.6 GB
==((====))==  Unsloth 2026.11.4: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

DeepSeek loaded for direct Kaggle inference (no training).


## `call_llm` / `extract_json` — the only place that talks to the model

`extract_json` strips a `<think>...</think>` block first (DeepSeek-R1 always reasons before answering, and a stray `{`/`[` inside its reasoning must not be mistaken for the real JSON), then robustly extracts just the JSON value even if the model adds commentary before or after it.

In [4]:
import json
import re

import requests


def call_llm(prompt):
    if RUNTIME == "kaggle":
        text = kaggle_tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                                     tokenize=False, add_generation_prompt=True)
        enc = kaggle_tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")
        with torch.inference_mode():
            out = kaggle_model.generate(**enc, max_new_tokens=KAGGLE_MAX_NEW_TOKENS,
                                        eos_token_id=KAGGLE_STOP_IDS,
                                        pad_token_id=kaggle_tokenizer.pad_token_id, do_sample=False,
                                        use_cache=True)
        return kaggle_tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                             skip_special_tokens=True)[0]
    if RUNTIME == "ollama":
        resp = requests.post(OLLAMA_URL, json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
                             timeout=180)
        resp.raise_for_status()
        return resp.json()["response"]
    return _mock_llm(prompt)


def _mock_llm(prompt):
    """Rough, generic-keyword stand-in for DeepSeek, dispatched by the TASK marker each prompt
    starts with - for a quick offline sanity check of the pipeline's control flow only. Its
    outputs are not meaningful compliance judgments; only RUNTIME='kaggle'/'ollama' are."""
    if prompt.startswith("TASK: classify"):
        return _mock_classify(prompt)
    if prompt.startswith("TASK: verdict"):
        return _mock_verdict(prompt)
    raise ValueError("Unrecognised prompt task for the mock LLM")


def extract_json(raw):
    """Robust JSON extraction for a real reasoning model's raw text: drop everything up to and
    including </think> first (if present), then use raw_decode (not json.loads) so trailing
    commentary after the JSON does not break parsing either."""
    if "</think>" in raw:
        raw = raw.split("</think>", 1)[1]
    start = min((i for i in (raw.find("{"), raw.find("[")) if i != -1), default=-1)
    if start == -1:
        raise ValueError(f"No JSON object/array found in model output: {raw!r}")
    return json.JSONDecoder().raw_decode(raw[start:])[0]


# Lightweight, dataset-agnostic topic detection, matched against a requirement's own text rather
# than a fixed category enum (the EU AI Act and GDPR sources use different taxonomies, so category
# names alone cannot be compared across them). Used by the deterministic evidence checks (cell 9)
# and by the mock classifier (cell 7, RUNTIME="mock" only).
def _topic_is_encryption(text):
    return re.search(r"encrypt|cryptograph", text, re.I) is not None


def _topic_is_access_control(text):
    return re.search(r"access[- ]control|least[- ]privilege|authoriz|authentic|role-based|RBAC",
                     text, re.I) is not None


TOPIC_MATCHERS = [("Encryption", _topic_is_encryption), ("AccessControl", _topic_is_access_control)]

## 1. Load ~300 real requirements (EU AI Act + GDPR), classify each (deterministic vs. llm_judgment)

In [5]:
import random
import re
from collections import Counter

from datasets import load_dataset

_ai_act_ds = load_dataset(AI_ACT_DATASET, "obligations", split="train")
_ai_act_df = _ai_act_ds.to_pandas()
_ai_act_sample = _ai_act_df.sample(n=min(MAX_AI_ACT_REQUIREMENTS, len(_ai_act_df)), random_state=42)
_ai_act_requirements = [{
    "id": row["obligation_id"],
    "regulation": "EU AI Act",
    "article": f"EU AI Act Art. {row['article']}" + (f" ({row['paragraph']})" if row["paragraph"] else ""),
    "category": row["obligation_type"],
    "risk_tier": row["risk_tier"],
    "text": row["summary"],
    "source_quote": row["quote"],
    "source_url": row["url"],
} for _, row in _ai_act_sample.iterrows()]

_gdpr_ds = load_dataset(GDPR_DATASET, split="train")
_gdpr_df = _gdpr_ds.to_pandas()
_gdpr_articles = _gdpr_df[_gdpr_df["type"] == "article"]
_gdpr_total_obligations = sum(len(obs) for obs in _gdpr_articles["key_obligations"])

_gdpr_requirements = []
for _, row in _gdpr_articles.iterrows():
    chapter_name = row["chapter"].split(" - ", 1)[-1] if row["chapter"] else "General"
    obligations = row["key_obligations"] if row["key_obligations"] is not None else []
    for j, obligation_text in enumerate(obligations):
        _gdpr_requirements.append({
            "id": f"gdpr-art{row['article_number']}-{j}",
            "regulation": "GDPR",
            "article": f"GDPR Art. {row['article_number']} ({row['title']})",
            "category": chapter_name,
            "risk_tier": None,
            "text": obligation_text,
            "source_quote": row["content"],
            "source_url": row["source_url"],
        })
random.Random(42).shuffle(_gdpr_requirements)
_gdpr_requirements = _gdpr_requirements[:MAX_GDPR_REQUIREMENTS]

RAW_REQUIREMENTS = _ai_act_requirements + _gdpr_requirements
_ids = [r["id"] for r in RAW_REQUIREMENTS]
assert len(_ids) == len(set(_ids)), "duplicate requirement ids across the combined dataset"

print(f"EU AI Act: {len(_ai_act_requirements)} atomic obligations sampled (of {len(_ai_act_df)} "
      f"available across {_ai_act_df['article'].nunique()} articles) from {AI_ACT_DATASET}")
print(f"GDPR:      {len(_gdpr_requirements)} atomic obligations sampled (of {_gdpr_total_obligations} "
      f"available across {len(_gdpr_articles)} articles) from {GDPR_DATASET}")
print(f"Total: {len(RAW_REQUIREMENTS)} real requirements - nothing hardcoded. Raise "
      f"MAX_AI_ACT_REQUIREMENTS / MAX_GDPR_REQUIREMENTS for more, up to {len(_ai_act_df)} + "
      f"{_gdpr_total_obligations} = {len(_ai_act_df) + _gdpr_total_obligations} available.")

(RUN_DIR / "regulations_dataset.json").write_text(json.dumps(RAW_REQUIREMENTS, indent=2), encoding="utf-8")
print(f"Saved -> {RUN_DIR / 'regulations_dataset.json'}")

CLASSIFY_PROMPT_TEMPLATE = '''TASK: classify
You are a compliance analyst. For each requirement below, decide whether it can be checked by a
deterministic pattern/rule in code (check_type="deterministic") or needs human-language judgment
(check_type="llm_judgment").

Reply with a JSON list, one object per requirement, in the same order, using its exact id:
[{{"id": "...", "check_type": "deterministic"|"llm_judgment"}}, ...]

REQUIREMENTS:
{requirements_block}
'''


def _chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def classify_requirements(raw_requirements):
    """Classified in chunks rather than one giant prompt - at ~300 requirements a single prompt
    would risk exceeding the model's context window, and one bad reply would lose everything."""
    classifications = {}
    chunks = list(_chunked(raw_requirements, CLASSIFY_BATCH_SIZE))
    for i, chunk in enumerate(chunks, 1):
        block = "\n".join(f"- {r['id']} ({r['regulation']} {r['article']}, category "
                          f"{r['category']}): {r['text']}" for r in chunk)
        prompt = CLASSIFY_PROMPT_TEMPLATE.format(requirements_block=block)
        try:
            answers = extract_json(call_llm(prompt))
            classifications.update({a["id"]: a.get("check_type", "llm_judgment") for a in answers})
        except Exception as exc:
            print(f"  [warn] classify chunk {i}/{len(chunks)} failed to parse ({exc}); "
                  f"defaulting its {len(chunk)} requirements to llm_judgment.")
            classifications.update({r["id"]: "llm_judgment" for r in chunk})
        print(f"  classified chunk {i}/{len(chunks)} ({len(chunk)} requirements)")
    return classifications


def _mock_classify(prompt):
    # Parsed back out of the prompt text itself (same contract as _mock_verdict), so this works
    # for whichever chunk it is handed rather than depending on outside state.
    classifications = []
    for line in prompt.splitlines():
        m = re.match(r"- (\S+) \(.*?\): (.*)", line)
        if not m:
            continue
        req_id, text = m.group(1), m.group(2)
        is_deterministic = any(matches(text) for _, matches in TOPIC_MATCHERS)
        classifications.append({"id": req_id,
                                "check_type": "deterministic" if is_deterministic else "llm_judgment"})
    return json.dumps(classifications)


_classifications = classify_requirements(RAW_REQUIREMENTS)
REQUIREMENTS = [{**r, "check_type": _classifications.get(r["id"], "llm_judgment")}
               for r in RAW_REQUIREMENTS]

print(f"\n{len(REQUIREMENTS)} requirements classified:")
for (reg, ct), n in sorted(Counter((r["regulation"], r["check_type"]) for r in REQUIREMENTS).items()):
    print(f"  {reg:12s} {ct:14s} {n}")

README.md: 0.00B [00:00, ?B/s]

data/obligations.parquet:   0%|          | 0.00/125k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/677 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/807 [00:00<?, ?B/s]

train-00000-of-00001.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/153 [00:00<?, ? examples/s]

EU AI Act: 200 atomic obligations sampled (of 677 available across 118 articles) from safelegalaidata/eu-ai-act-structured
GDPR:      100 atomic obligations sampled (of 152 available across 57 articles) from AYI-NEDJIMI/gdpr-en
Total: 300 real requirements - nothing hardcoded. Raise MAX_AI_ACT_REQUIREMENTS / MAX_GDPR_REQUIREMENTS for more, up to 677 + 152 = 829 available.
Saved -> agentic_run/regulations_dataset.json
  [warn] classify chunk 1/12 failed to parse (No JSON object/array found in model output: '<think>\nFirst, I am considering the task. I\'m a compliance analyst, and I need to classify each requirement as either "deterministic" or "llm_judgment". Deterministic means it can be checked by a rule in code, like a pattern or a straightforward condition. LLM_judgment means it requires human-language judgment, probably because it involves interpretation or subjective elements.\n\nI need to reply with a JSON list, one object per requirement, in the same order, using the exact id pr

## 2. Collect evidence from a real codebase

- **llm_judgment** requirements: TF-IDF + cosine similarity retrieval (a small, real RAG) over real files from `llm-sast-v1` (sampled from 3,055 distinct repos, 18 languages/IaC types).
- **deterministic** requirements: a generic regex check for the two topics most amenable to pattern matching (encryption, access control), detected from the requirement's own text rather than a fixed category - the EU AI Act and GDPR sources use different taxonomies, so category names alone cannot be compared across them. On arbitrary real code this is necessarily approximate - unlike the old synthetic data, there is no clean single toggle variable to match everywhere, so "no match" correctly falls back to `insufficient_evidence`, not a guess.

In [6]:
import re

import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# The dataset is not pre-shuffled (rows are grouped by source repo), so a naive train[:N] slice
# landed almost entirely in a single repo - shuffle first for genuine cross-repo diversity.
_evidence_ds = load_dataset(EVIDENCE_DATASET, split="train").shuffle(seed=42).select(range(EVIDENCE_LIMIT))
df = _evidence_ds.to_pandas()
# The raw columns are just "messages" + "_meta" (a dict) - file_path/source_repo/classification live
# inside _meta, not as their own top-level columns.
df["content"] = df["messages"].apply(lambda m: m[1]["content"])  # the user turn: "FILE: ...\n...code..."
df["path"] = df["_meta"].apply(lambda m: m["file_path"])
df["source_repo"] = df["_meta"].apply(lambda m: m["source_repo"])
df["classification"] = df["_meta"].apply(lambda m: m["classification"])
print(f"Evidence corpus: {len(df)} real files (of 19,024 total in {EVIDENCE_DATASET}), "
      f"{df['source_repo'].nunique()} distinct repos, languages: "
      f"{sorted(df['classification'].unique())}")


def _check_encryption(code_text):
    if re.search(r"encrypt(ed|ion)?\s*[:=]\s*(false|False|\"?no\"?)|insecure\s*[:=]\s*true|"
                r"ssl\s*[:=]\s*false", code_text, re.I):
        return "fail"
    if re.search(r"encrypt(ed|ion)?\s*[:=]\s*(true|True)|kms_key|sse_algorithm", code_text, re.I):
        return "pass"
    return None


def _check_access_control(code_text):
    if re.search(r"0\.0\.0\.0/0|public[-_]?read|allow.*\*.*\*|AllowAnyOrigin", code_text):
        return "fail"
    if re.search(r"least[-_ ]privilege|principal.*arn:aws|RBAC|role[-_]binding", code_text, re.I):
        return "pass"
    return None


DETERMINISTIC_CHECKS = {"Encryption": _check_encryption, "AccessControl": _check_access_control}

_vectorizer = TfidfVectorizer(stop_words="english", max_features=20000)
_doc_vectors = _vectorizer.fit_transform(df["content"])


def retrieve_evidence(requirement_text, top_k=TOP_K_EVIDENCE):
    """TF-IDF + cosine similarity retrieval - the RAG step, over the real evidence corpus."""
    query_vec = _vectorizer.transform([requirement_text])
    scores = cosine_similarity(query_vec, _doc_vectors)[0]
    top_idx = scores.argsort()[::-1][:top_k]
    return [{"row": int(i), "path": df.loc[i, "path"], "content": df.loc[i, "content"],
             "score": round(float(scores[i]), 4)} for i in top_idx if scores[i] > 0]


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/37.9M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.71M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15219 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1902 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1903 [00:00<?, ? examples/s]

Evidence corpus: 500 real files (of 19,024 total in aioutfitters/llm-sast-v1), 292 distinct repos, languages: ['arm_template', 'cloudformation', 'docker_compose', 'dockerfile', 'github_actions', 'gitlab_ci', 'go', 'helm', 'java', 'javascript', 'jenkinsfile', 'kubernetes', 'php', 'python', 'ruby', 'rust', 'terraform', 'typescript']


## 3. Verdict, with an anti-hallucination guardrail

Judged in small batches (`VERDICT_BATCH_SIZE`), not one LLM call per requirement - at a few hundred requirements, one call each would make a Kaggle run impractically slow. Each requirement only ever "sees" its own retrieved evidence inside the batch prompt, and the guardrail checks every citation against that same per-requirement set, so batching cannot let a citation leak between requirements.

In [7]:
import re

# Generic security/compliance cues that plausibly show up across diverse real code - a rough
# approximation, only used for RUNTIME="mock".
_COMPLIANT_CUES = ["encrypted", "least privilege", "rbac", "audit_log", "logger.", "consent"]
_NONCOMPLIANT_CUES = ["0.0.0.0/0", "public-read", "hardcoded", "password =", "plaintext"]

BATCH_VERDICT_PROMPT_TEMPLATE = '''TASK: verdict_batch
You are a compliance auditor. For EACH requirement below, decide whether its OWN evidence block
satisfies it. A citation is only valid if it names a row number shown in THAT SAME requirement's
evidence block - never reuse a row shown for a different requirement.

Reply with a JSON list, one object per requirement, in the same order, using its exact id:
[{{"id": "...", "status": "compliant"|"non_compliant"|"insufficient_evidence",
  "cited_row": <int or null>, "rationale": "<short, specific explanation>"}}, ...]

{items_block}
'''


def _format_verdict_item(requirement, evidence_rows):
    evidence_block = "\n".join(f"  [row {e['row']}] {e['path']}: {e['content'][:500]!r}"
                               for e in evidence_rows)
    return (f'Requirement {requirement["id"]} ({requirement["article"]}): {requirement["text"]}\n'
           f'Evidence:\n{evidence_block}')


def _mock_verdict(prompt):
    answers = []
    for block in prompt.split("Requirement ")[1:]:
        req_id = block.split(" ", 1)[0]
        status, cited_row = "insufficient_evidence", None
        for line in block.splitlines():
            m = re.match(r"\s*\[row (\d+)\]", line)
            if not m:
                continue
            low = line.lower()
            if any(cue in low for cue in _NONCOMPLIANT_CUES):
                status, cited_row = "non_compliant", int(m.group(1))
                break
            if any(cue in low for cue in _COMPLIANT_CUES):
                status, cited_row = "compliant", int(m.group(1))
                break
        answers.append({"id": req_id, "status": status, "cited_row": cited_row, "rationale": "mock"})
    return json.dumps(answers)


def judge_requirements_batch(items, checkpoint_path=None):
    """items: list of (requirement, evidence_rows) pairs that need an LLM judgment call (evidence_rows
    is non-empty - requirements with no evidence are resolved without calling the model at all).
    Judged VERDICT_BATCH_SIZE at a time: one call per chunk instead of one call per requirement, so a
    200-400 requirement run stays within a realistic number of slow, reasoning-model GPU calls."""
    results = {}
    chunks = list(_chunked(items, VERDICT_BATCH_SIZE))
    for i, chunk in enumerate(chunks, 1):
        valid_rows_by_id = {r["id"]: {e["row"] for e in ev} for r, ev in chunk}
        prompt = BATCH_VERDICT_PROMPT_TEMPLATE.format(
            items_block="\n\n".join(_format_verdict_item(r, ev) for r, ev in chunk))
        try:
            answers = extract_json(call_llm(prompt))
            answers_by_id = {a["id"]: a for a in answers if isinstance(a, dict) and "id" in a}
        except Exception as exc:
            print(f"  [warn] verdict batch {i}/{len(chunks)} failed to parse ({exc}); "
                  f"marking its {len(chunk)} requirements insufficient_evidence.")
            answers_by_id = {}
            if RUNTIME == "kaggle":
                torch.cuda.empty_cache()
        for r, ev in chunk:
            ans = answers_by_id.get(r["id"])
            if ans is None:
                results[r["id"]] = {"status": "insufficient_evidence", "cited_row": None,
                                    "rationale": "Model did not return a parseable verdict for this id."}
                continue
            cited_row = ans.get("cited_row")
            # Anti-hallucination guardrail: the cited row must be one THIS requirement's own
            # evidence actually contained, not merely any row shown anywhere in the batch.
            if cited_row is not None and cited_row not in valid_rows_by_id[r["id"]]:
                results[r["id"]] = {"status": "insufficient_evidence", "cited_row": None,
                                    "rationale": f"Verdict cited row {cited_row}, which was never "
                                                 f"shown for this requirement - discarded as a "
                                                 f"hallucinated citation."}
            else:
                results[r["id"]] = {"status": ans.get("status", "insufficient_evidence"),
                                    "cited_row": cited_row, "rationale": ans.get("rationale", "")}
        if checkpoint_path is not None:
            checkpoint_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
        print(f"  judged batch {i}/{len(chunks)} ({len(chunk)} requirements) - checkpoint saved")
    return results


def run_deterministic_check(requirement, evidence_rows):
    topic = next((label for label, matches in TOPIC_MATCHERS if matches(requirement["text"])), None)
    check_fn = DETERMINISTIC_CHECKS.get(topic)
    if check_fn is None:
        return {"status": "insufficient_evidence", "cited_row": None,
               "rationale": "No deterministic check implemented for this requirement's topic."}
    for e in evidence_rows:
        result = check_fn(e["content"])
        if result == "fail":
            return {"status": "non_compliant", "cited_row": e["row"],
                   "rationale": "Deterministic rule matched a violation pattern."}
        if result == "pass":
            return {"status": "compliant", "cited_row": e["row"],
                   "rationale": "Deterministic rule matched a compliant pattern."}
    return {"status": "insufficient_evidence", "cited_row": None,
           "rationale": "No retrieved row matched the deterministic rule's patterns."}

## 4. Compile the audit report

Deterministic checks and "no evidence retrieved" cases are resolved first without touching the model; only genuine `llm_judgment` cases with retrieved evidence go through `judge_requirements_batch`, which checkpoints its progress to `verdicts_checkpoint.json` after every batch.

In [8]:
import time

_t0 = time.time()
evidence_by_id = {}
results_by_id = {}
llm_items = []

for req in REQUIREMENTS:
    evidence_rows = retrieve_evidence(req["text"])
    evidence_by_id[req["id"]] = evidence_rows
    if req["check_type"] == "deterministic":
        results_by_id[req["id"]] = run_deterministic_check(req, evidence_rows)
    elif evidence_rows:
        llm_items.append((req, evidence_rows))
    else:
        results_by_id[req["id"]] = {"status": "insufficient_evidence", "cited_row": None,
                                    "rationale": "No evidence retrieved."}

print(f"Evidence retrieved for all {len(REQUIREMENTS)} requirements: {len(llm_items)} need an LLM "
      f"judgment call, {len(REQUIREMENTS) - len(llm_items)} already resolved (deterministic rule or "
      f"no evidence at all) without calling the model.")

results_by_id.update(judge_requirements_batch(llm_items, checkpoint_path=RUN_DIR / "verdicts_checkpoint.json"))

report_rows = [{**req, **results_by_id[req["id"]], "n_evidence_considered": len(evidence_by_id[req["id"]])}
               for req in REQUIREMENTS]

report_df = pd.DataFrame(report_rows)
display(report_df[["id", "regulation", "article", "category", "check_type", "status", "cited_row",
                   "n_evidence_considered", "rationale"]])

(RUN_DIR / "audit_report.json").write_text(json.dumps(report_rows, indent=2), encoding="utf-8")
print(f"\nWritten: {RUN_DIR / 'audit_report.json'}  ({time.time() - _t0:.0f}s total)")
print(f"\nStatus counts: {report_df['status'].value_counts().to_dict()}")
print(f"Per-regulation status:\n{report_df.groupby('regulation')['status'].value_counts()}")

Evidence retrieved for all 300 requirements: 162 need an LLM judgment call, 138 already resolved (deterministic rule or no evidence at all) without calling the model.
  [warn] verdict batch 1/41 failed to parse (Expecting value: line 1 column 2 (char 1)); marking its 4 requirements insufficient_evidence.
  judged batch 1/41 (4 requirements) - checkpoint saved
  judged batch 2/41 (4 requirements) - checkpoint saved
  judged batch 3/41 (4 requirements) - checkpoint saved
  judged batch 4/41 (4 requirements) - checkpoint saved
  judged batch 5/41 (4 requirements) - checkpoint saved
  judged batch 6/41 (4 requirements) - checkpoint saved
  [warn] verdict batch 7/41 failed to parse (Expecting value: line 1 column 2 (char 1)); marking its 4 requirements insufficient_evidence.
  judged batch 7/41 (4 requirements) - checkpoint saved
  judged batch 8/41 (4 requirements) - checkpoint saved
  judged batch 9/41 (4 requirements) - checkpoint saved
  judged batch 10/41 (4 requirements) - checkpoint 

,id,regulation,article,category,check_type,status,cited_row,n_evidence_considered,rationale
0,art-106-0-amendment,EU AI Act,EU AI Act Art. 106,amendment-of-other-act,llm_judgment,insufficient_evidence,None,3,Model did not return a parseable verdict for t...
1,art-54-5-6,EU AI Act,EU AI Act Art. 54 (5),derogation-or-exemption,llm_judgment,insufficient_evidence,None,3,Model did not return a parseable verdict for t...
2,art-6-3-annex-iii-derogation,EU AI Act,EU AI Act Art. 6 (3),derogation-or-exemption,llm_judgment,insufficient_evidence,None,3,Model did not return a parseable verdict for t...
3,art-57-14,EU AI Act,EU AI Act Art. 57 (14),governance-institutional,llm_judgment,insufficient_evidence,None,3,Model did not return a parseable verdict for t...
4,art-13-1-transparent-operation,EU AI Act,EU AI Act Art. 13 (1),transparency,llm_judgment,insufficient_evidence,None,3,The provided evidence does not contain any cod...
...,...,...,...,...,...,...,...,...,...
295,gdpr-art35-0,GDPR,GDPR Art. 35 (Data protection impact assessmen...,Controller and Processor,deterministic,insufficient_evidence,None,3,No deterministic check implemented for this re...
296,gdpr-art9-1,GDPR,GDPR Art. 9 (Processing of special categories ...,Principles,llm_judgment,non_compliant,None,3,The evidence block does not contain any code t...
297,gdpr-art29-0,GDPR,GDPR Art. 29 (Processing under the authority o...,Controller and Processor,deterministic,insufficient_evidence,None,3,No deterministic check implemented for this re...
298,gdpr-art17-2,GDPR,GDPR Art. 17 (Right to erasure (right to be fo...,Rights of the Data Subject,llm_judgment,non_compliant,None,3,The evidence block does not contain any code t...



Written: agentic_run/audit_report.json  (5528s total)

Status counts: {'insufficient_evidence': 275, 'non_compliant': 25}
Per-regulation status:
regulation  status               
EU AI Act   insufficient_evidence    177
            non_compliant             23
GDPR        insufficient_evidence     98
            non_compliant              2
Name: count, dtype: int64
